# Kommentar-Emotionen mit GoEmotions

Dieses Notebook analysiert die Emotionen in TikTok-Kommentaren mit einem RoBERTa-basierten GoEmotions-Modell.

Ziele:
- dominante Kommentar-Emotion pro Kommentar bestimmen
- 28 Emotionslabels plus Score extrahieren
- KI- und Real-Kommentare auf Kommentar- und Video-Ebene vergleichen
- den Zusammenhang zwischen Emotionsprofil und Engagement prüfen


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, pipeline



In [2]:
BASE_DIR = Path.cwd().resolve().parents[1]
COMMENTS_PATH = BASE_DIR / "data" / "01_raw" / "comments_metadata" / "02_AI_AND_REAL_TIKTOK_COMMENTS_DATASET_2025.csv"
VIDEOS_PATH = BASE_DIR / "data" / "01_raw" / "videos_metadata" / "01_AI_AND_REAL_TIKTOK_VIDEOS_DATASET_2025.csv"
VIDEO_DATA_PATH = VIDEOS_PATH
AI_VIDEOS_RAW_PATH = BASE_DIR / "data" / "01_raw" / "videos_metadata" / "01_AI_TIKTOK_VIDEOS_DATASET_2025.csv"
REAL_VIDEOS_RAW_PATH = BASE_DIR / "data" / "01_raw" / "videos_metadata" / "01_REAL_TIKTOK_VIDEOS_DATASET_2025.csv"
OUTPUT_DIR = BASE_DIR / "comments" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLUMN = "comment_text"
FILTER_ENGLISH_ONLY = True
KEEP_REPLIES = True
BATCH_SIZE = 32
MAX_LENGTH = 512
EMOTION_THRESHOLD = 0.10

MODEL_NAME = "SamLowe/roberta-base-go_emotions"
COMMENT_RESULTS_CSV = OUTPUT_DIR / "03_comment_emotion_results.csv"
VIDEO_LEVEL_CSV = OUTPUT_DIR / "03_comment_emotion_video_level.csv"
MISSING_ENGAGEMENT_EXCLUDED_CSV = OUTPUT_DIR / "03_comment_emotion_excluded_missing_engagement.csv"
MISSING_VIDEO_METADATA_CSV = OUTPUT_DIR / "03_comment_emotion_missing_video_metadata.csv"
ENGAGEMENT_SOURCE_CSV = OUTPUT_DIR / "03_comment_emotion_engagement_sources.csv"
print(f"Kommentare: {COMMENTS_PATH}")
print(f"Ausgabe: {OUTPUT_DIR}")
print(f"Modell: {MODEL_NAME}")
print(f"Emotion-Threshold für Emotionsvielfalt: {EMOTION_THRESHOLD}")


Kommentare: /Users/hannahernstberger/Documents/Master_/data/01_raw/comments_metadata/02_AI_AND_REAL_TIKTOK_COMMENTS_DATASET_2025.csv
Ausgabe: /Users/hannahernstberger/Documents/Master_/comments/results
Modell: SamLowe/roberta-base-go_emotions
Emotion-Threshold für Emotionsvielfalt: 0.1


In [3]:
keep_cols = [
    "comment_id",
    "video_id",
    "influencer_type",
    "comment_language",
    "is_reply_to_comment",
    "comment_like_count",
    TEXT_COLUMN,
]
comments_df = pd.read_csv(COMMENTS_PATH, usecols=keep_cols).copy()
initial_n = len(comments_df)

# Engagement wird zuerst aus dem kombinierten Video-Datensatz gelesen.
# Fehlende Video-IDs werden danach aus den Roh-Video-Dateien rekonstruiert.
comments_df["video_id"] = comments_df["video_id"].astype("string").str.strip()
video_meta = pd.read_csv(
    VIDEO_DATA_PATH,
    usecols=["video_id", "video_engagement_rate"],
).drop_duplicates(subset="video_id")
video_meta["video_id"] = video_meta["video_id"].astype("string").str.strip()
video_meta["engagement_source"] = "combined_video_dataset"

raw_video_meta_frames = []
for raw_path, raw_type in [(AI_VIDEOS_RAW_PATH, "ai_raw_video_dataset"), (REAL_VIDEOS_RAW_PATH, "real_raw_video_dataset")]:
    raw_meta = pd.read_csv(raw_path, usecols=["id", "likes", "comments", "shares", "plays"])
    raw_meta = raw_meta.rename(columns={"id": "video_id"})
    raw_meta["video_id"] = raw_meta["video_id"].astype("string").str.strip()
    for metric_col in ["likes", "comments", "shares", "plays"]:
        raw_meta[metric_col] = pd.to_numeric(raw_meta[metric_col], errors="coerce")
    raw_meta["video_engagement_rate"] = (
        raw_meta["likes"].fillna(0) + raw_meta["comments"].fillna(0) + raw_meta["shares"].fillna(0)
    ) / raw_meta["plays"].replace(0, np.nan)
    raw_meta["engagement_source"] = raw_type
    raw_video_meta_frames.append(raw_meta[["video_id", "video_engagement_rate", "engagement_source"]])
raw_video_meta = pd.concat(raw_video_meta_frames, ignore_index=True).drop_duplicates(subset="video_id")

video_meta_complete = pd.concat([video_meta, raw_video_meta], ignore_index=True)
video_meta_complete = video_meta_complete.dropna(subset=["video_engagement_rate"])
video_meta_complete = video_meta_complete.drop_duplicates(subset="video_id", keep="first")
comments_df = comments_df.merge(video_meta_complete, on="video_id", how="left")
merged_n = len(comments_df)

comments_df = comments_df.drop_duplicates(subset="comment_id").copy()
dedup_n = len(comments_df)
comments_df[TEXT_COLUMN] = (
    comments_df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
comments_df = comments_df[comments_df[TEXT_COLUMN] != ""].copy()
after_text_clean_n = len(comments_df)
if FILTER_ENGLISH_ONLY:
    comments_df = comments_df[comments_df["comment_language"].fillna("").str.lower().eq("en")].copy()
after_language_filter_n = len(comments_df)
if not KEEP_REPLIES:
    comments_df = comments_df[comments_df["is_reply_to_comment"].fillna("no").str.lower().ne("yes")].copy()
after_reply_filter_n = len(comments_df)

missing_video_metadata_df = (
    comments_df[comments_df["video_engagement_rate"].isna()]
    .groupby(["video_id", "influencer_type"], dropna=False)
    .agg(
        comment_count=("comment_id", "count"),
        example_comment=(TEXT_COLUMN, "first"),
    )
    .reset_index()
    .sort_values("comment_count", ascending=False)
)
excluded_missing_engagement_df = comments_df[comments_df["video_engagement_rate"].isna()].copy()
comments_df = comments_df[comments_df["video_engagement_rate"].notna()].copy()
after_engagement_filter_n = len(comments_df)
missing_engagement_before_filter = len(excluded_missing_engagement_df)
missing_engagement_video_n = excluded_missing_engagement_df["video_id"].nunique()
missing_engagement = comments_df["video_engagement_rate"].isna().sum()
engagement_source_video_df = (
    comments_df.groupby(["video_id", "influencer_type", "engagement_source", "video_engagement_rate"], dropna=False)
    .size()
    .rename("comment_count")
    .reset_index()
    .sort_values(["engagement_source", "comment_count"], ascending=[True, False])
)

comments_df["Typ"] = comments_df["influencer_type"].map({"ai": "KI", "real": "Real"}).fillna(comments_df["influencer_type"])
comments_df["comment_length_chars"] = comments_df[TEXT_COLUMN].str.len()
comments_df["comment_length_words"] = comments_df[TEXT_COLUMN].str.split().str.len()

prep_overview = pd.DataFrame([
    ("Ausgangsdatensatz", initial_n),
    ("Nach Engagement-Merge", merged_n),
    ("Nach Entfernen doppelter comment_id", dedup_n),
    ("Nach Entfernen leerer Kommentare", after_text_clean_n),
    ("Nach Sprachfilter (nur Englisch)" if FILTER_ENGLISH_ONLY else "Ohne Sprachfilter", after_language_filter_n),
    ("Replies beibehalten" if KEEP_REPLIES else "Nach Reply-Filter", after_reply_filter_n),
    ("Ausgeschlossen: keine Engagement-Rate", missing_engagement_before_filter),
    ("Videos ohne gematchte Engagement-Rate", missing_engagement_video_n),
    ("Finale Analyse mit Engagement-Rate", after_engagement_filter_n),
], columns=["Schritt", "Kommentare"])

print(f"Kommentare nach Preprocessing: {len(comments_df):,}")
print(f"Ausgeschlossen wegen fehlender Engagement-Rate: {missing_engagement_before_filter:,} Kommentare aus {missing_engagement_video_n:,} Videos")
print(f"Kommentare ohne gematchte Engagement-Rate nach Filter: {missing_engagement:,}")
display(prep_overview)
display(comments_df["engagement_source"].value_counts().rename_axis("engagement_source").reset_index(name="comments"))
display(missing_video_metadata_df.head(20))
display(comments_df.head(3))


Kommentare nach Preprocessing: 32,671
Ausgeschlossen wegen fehlender Engagement-Rate: 6,944 Kommentare aus 83 Videos
Kommentare ohne gematchte Engagement-Rate nach Filter: 0


,Schritt,Kommentare
0,Ausgangsdatensatz,80817
1,Nach Engagement-Merge,80817
2,Nach Entfernen doppelter comment_id,80817
3,Nach Entfernen leerer Kommentare,80497
4,Nach Sprachfilter (nur Englisch),39615
5,Replies beibehalten,39615
6,Ausgeschlossen: keine Engagement-Rate,6944
7,Videos ohne gematchte Engagement-Rate,83
8,Finale Analyse mit Engagement-Rate,32671


,engagement_source,comments
0,combined_video_dataset,30967
1,ai_raw_video_dataset,1466
2,real_raw_video_dataset,238


,video_id,influencer_type,comment_count,example_comment
3,7208351443193974062,real,1787,If Niall Horan looked at me like that I’d forg...
6,7371613331704139040,real,1412,Julia Zelg for me. I couldn’t recognize her wh...
2,7040984071631228206,real,853,So y’all rich rich. Huh. Are you looking to ad...
1,7014140703680826629,real,701,I don’t have a nice side profile like u girl :/
15,7538581166312426774,real,374,"go to David's salon, your $3 would be so much ..."
4,7320000759762881824,real,357,where is your coat from
25,7578555804157103391,real,179,Why are people bullying a child who is just ha...
7,7383032827526450478,real,170,books and pilates are real hobbies don’t let m...
5,7364519121855319339,real,62,this would heal me
0,6869412214848163077,real,61,the fact that this has 6.7M likes


,comment_id,comment_text,comment_like_count,video_id,is_reply_to_comment,comment_language,influencer_type,video_engagement_rate,engagement_source,Typ,comment_length_chars,comment_length_words
1178,7541804977875550983,Wow I liked 💗,1,7541766006897773838,no,en,real,0.020297,combined_video_dataset,Real,13,4
1192,7543754834214945557,Need this therapy too✨,0,7541766006897773838,no,en,real,0.020297,combined_video_dataset,Real,22,4
1197,7572978399367676686,and the little sister too,1,7571506841398562078,yes,en,real,0.046320,combined_video_dataset,Real,25,5


In [4]:
if torch.cuda.is_available():
    pipeline_device = 0
    device_label = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    pipeline_device = "mps"
    device_label = "mps"
else:
    pipeline_device = -1
    device_label = "cpu"

print(f"Verwendetes Device für Emotionserkennung: {device_label}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, model_max_length=MAX_LENGTH)
emotion_pipe = pipeline(
    task="text-classification",
    model=MODEL_NAME,
    tokenizer=tokenizer,
    device=pipeline_device,
    truncation=True,
    max_length=MAX_LENGTH,
    top_k=None,
)

texts = comments_df[TEXT_COLUMN].tolist()
all_results = []
for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="GoEmotions"):
    batch = texts[start:start + BATCH_SIZE]
    batch_results = emotion_pipe(batch, truncation=True, padding=True, max_length=MAX_LENGTH)
    all_results.extend(batch_results)

label_order = sorted({item["label"] for result in all_results for item in result})
rows = []
for text, result in zip(texts, all_results):
    scores = {entry["label"]: entry["score"] for entry in result}
    dominant = max(scores, key=scores.get)
    row = {
        TEXT_COLUMN: text,
        "comment_emotion_label": dominant,
        "comment_emotion_score": scores.get(dominant, 0.0),
        "comment_emotion_unique_labels": int(sum(score >= EMOTION_THRESHOLD for score in scores.values())),
    }
    for label in label_order:
        row[f"emotion_{label}"] = scores.get(label, 0.0)
    rows.append(row)

emotion_df = pd.DataFrame(rows)
comments_df = pd.concat([comments_df.reset_index(drop=True), emotion_df], axis=1)


def dominant_label(series):
    mode = series.mode(dropna=True)
    return mode.iloc[0] if not mode.empty else np.nan


video_level_df = (
    comments_df.groupby(["video_id", "Typ", "video_engagement_rate"], dropna=False)
    .agg(
        comment_count=("comment_id", "count"),
        dominant_comment_emotion=("comment_emotion_label", dominant_label),
        emotion_unique_labels_mean=("comment_emotion_unique_labels", "mean"),
        emotion_unique_labels_median=("comment_emotion_unique_labels", "median"),
    )
    .reset_index()
)

annotation_columns = [
    "comment_emotion_label",
    "comment_emotion_score",
    "comment_emotion_unique_labels",
] + [f"emotion_{label}" for label in label_order]

print(f"Analysierte Kommentare: {len(comments_df):,}")
print(f"Hinzugefuegte Spalten: {annotation_columns[:3]} + {len(label_order)} Emotionsscore-Spalten")
preview_columns = [
    "comment_id",
    TEXT_COLUMN,
    "comment_emotion_label",
    "comment_emotion_score",
    "comment_emotion_unique_labels",
] + [f"emotion_{label}" for label in label_order[:5]]

display(comments_df[preview_columns].head(10))
display(pd.DataFrame({"added_column": annotation_columns}).head(40))



Verwendetes Device für Emotionserkennung: mps


Device set to use mps


GoEmotions:   0%|          | 0/1021 [00:00<?, ?it/s]

Analysierte Kommentare: 32,671
Hinzugefuegte Spalten: ['comment_emotion_label', 'comment_emotion_score', 'comment_emotion_unique_labels'] + 28 Emotionsscore-Spalten


,comment_id,comment_text,comment_text,comment_emotion_label,comment_emotion_score,comment_emotion_unique_labels,emotion_admiration,emotion_amusement,emotion_anger,emotion_annoyance,emotion_approval
0,7541804977875550983,Wow I liked 💗,Wow I liked 💗,love,0.798831,3,0.136194,0.009749,0.003312,0.004332,0.028593
1,7543754834214945557,Need this therapy too✨,Need this therapy too✨,neutral,0.326392,3,0.004874,0.000571,0.003052,0.013947,0.187767
2,7572978399367676686,and the little sister too,and the little sister too,neutral,0.967026,1,0.002811,0.001496,0.001771,0.005286,0.017554
3,7553098989831308087,yup that is me and my future wife...lol 🤣🤣,yup that is me and my future wife...lol 🤣🤣,amusement,0.942482,1,0.017672,0.942482,0.001311,0.003702,0.017005
4,7553627849786852126,Is he bothering you queen?,Is he bothering you queen?,curiosity,0.593157,3,0.001648,0.002091,0.008518,0.030790,0.006075
5,7553096163727622923,Teamwork,Teamwork,neutral,0.963178,1,0.005158,0.001690,0.002056,0.005024,0.021152
6,7553128019142116127,Was kind hoping he hold the dish with his butt...,Was kind hoping he hold the dish with his butt...,optimism,0.591021,2,0.002286,0.003084,0.003158,0.011352,0.007637
7,7560097482052076310,I got a life changing businessproposal for you...,I got a life changing businessproposal for you...,neutral,0.339719,4,0.014108,0.003435,0.001528,0.004213,0.128441
8,7556426361139135262,Where is that at ??,Where is that at ??,curiosity,0.493797,3,0.001225,0.001498,0.002608,0.009929,0.004702
9,7549955305976611602,Oooo you definitely slayed Queen! 😋,Oooo you definitely slayed Queen! 😋,anger,0.477715,3,0.009276,0.018940,0.477715,0.255736,0.006127


,added_column
0,comment_emotion_label
1,comment_emotion_score
2,comment_emotion_unique_labels
3,emotion_admiration
4,emotion_amusement
5,emotion_anger
6,emotion_annoyance
7,emotion_approval
8,emotion_caring
9,emotion_confusion


In [ ]:
comments_df.to_csv(COMMENT_RESULTS_CSV, index=False)
video_level_df.to_csv(VIDEO_LEVEL_CSV, index=False)
excluded_missing_engagement_df.to_csv(MISSING_ENGAGEMENT_EXCLUDED_CSV, index=False)
missing_video_metadata_df.to_csv(MISSING_VIDEO_METADATA_CSV, index=False)
engagement_source_video_df.to_csv(ENGAGEMENT_SOURCE_CSV, index=False)
print(f"Kommentare mit Emotionen gespeichert: {COMMENT_RESULTS_CSV.name}")
print(f"Video-Level-Datei gespeichert: {VIDEO_LEVEL_CSV.name}")


Kommentare mit Emotionen gespeichert: 03_comment_emotion_results.csv
Video-Level-Datei gespeichert: 03_comment_emotion_video_level.csv
